In [ ]:
import importlib.util
import shutil
import subprocess
import sys
import json
from pathlib import Path

# ------------------------------------------------------------
# Detect Google Colab
# ------------------------------------------------------------
try:
    import google.colab  # type: ignore
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print(f"Environment: {'Google Colab' if IS_COLAB else 'Local Jupyter / Python'}")

# ------------------------------------------------------------
# Define the security_frameworks directory
# ------------------------------------------------------------
if IS_COLAB:
    SECURITY_FRAMEWORKS_DIR = Path("/content/security_frameworks")
else:
    SECURITY_FRAMEWORKS_DIR = Path.cwd()
    # Guard: local runs must be launched from the security_frameworks folder
    # that actually holds this notebook, otherwise OUTPUT_DIR would land in
    # the wrong place.
    if not (SECURITY_FRAMEWORKS_DIR / "SSP_exploration.ipynb").exists():
        raise RuntimeError(
            "Run this notebook with its working directory set to the "
            "security_frameworks folder that contains SSP_exploration.ipynb. "
            f"Current working directory is: {SECURITY_FRAMEWORKS_DIR}"
        )

SECURITY_FRAMEWORKS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Repository and output locations
# ------------------------------------------------------------
OUTPUT_DIR = SECURITY_FRAMEWORKS_DIR / "SSP_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Security frameworks directory: {SECURITY_FRAMEWORKS_DIR}")
print(f"Output directory:              {OUTPUT_DIR}")

# ------------------------------------------------------------
# Check/install dependencies
# ------------------------------------------------------------
if importlib.util.find_spec("requests") is None:
    print("\nInstalling requests...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"], check=True)

if importlib.util.find_spec("bs4") is None:
    print("\nInstalling beautifulsoup4...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "beautifulsoup4"], check=True)

print("Environment and dependencies ready.")


In [ ]:
import time
import re
import urllib.request
import urllib.error
from urllib.parse import urljoin
from bs4 import BeautifulSoup

BASE_URL = "https://info.standards.tech.gov.sg/ssp/"
CATALOG_BASE = "https://info.standards.tech.gov.sg/control-catalog/"

REQUEST_TIMEOUT = 15
MAX_RETRIES = 3
RETRY_BACKOFF = 1.5  # seconds, doubles each retry
REQUEST_DELAY = 0.5  # polite delay between requests

_stats = {"requests": 0, "not_found": 0}

def fetch_html(url, allow_404=True):
    """Fetch a URL's HTML.

    Retries transient failures up to MAX_RETRIES times with exponential
    backoff. A 404 is not retried: if allow_404, it is counted and None is
    returned; otherwise it raises like any other failure. Any other
    non-recoverable failure raises RuntimeError after retries are exhausted.
    """
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) SSP-Explorer/1.0"},
    )
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            time.sleep(REQUEST_DELAY)
            _stats["requests"] += 1
            with urllib.request.urlopen(req, timeout=REQUEST_TIMEOUT) as response:
                return response.read().decode("utf-8")
        except urllib.error.HTTPError as e:
            if e.code == 404:
                if allow_404:
                    _stats["not_found"] += 1
                    return None
                raise RuntimeError(f"404 fetching required page {url}") from e
            last_error = e
        except Exception as e:  # noqa: BLE001 - network errors of any kind
            last_error = e

        if attempt < MAX_RETRIES:
            sleep_for = RETRY_BACKOFF * (2 ** (attempt - 1))
            print(f"  [retry {attempt}/{MAX_RETRIES}] {url} failed ({last_error}); retrying in {sleep_for:.1f}s")
            time.sleep(sleep_for)

    raise RuntimeError(f"Failed to fetch {url} after {MAX_RETRIES} attempts: {last_error}")


def leaf_blocks(container):
    """Yield (tag_name, element) for the container's descendant elements
    that themselves contain no element children -- i.e. the actual text
    landmarks of the page, regardless of whether the site wraps them in
    extra <div>s."""
    for el in container.find_all(True):
        if el.find(True) is not None:
            continue
        yield el.name, el


CONTROL_ID_RE = re.compile(r"^([A-Z]{2,3})-(\d+):\s*(.+)$")
SECTION_LABELS = {
    "control statement": "statement",
    "control recommendations": "recommendations",
    "risk statement": "risk",
    "rationale": "rationale",
    "parameters": "parameters",
}


def parse_parameters_table(table_tag):
    rows = []
    trs = table_tag.find_all("tr")
    if not trs:
        return rows
    header_cells = [c.get_text(strip=True).lower() for c in trs[0].find_all(["th", "td"])]
    for tr in trs[1:]:
        cells = [c.get_text(strip=True, separator=" ") for c in tr.find_all(["td", "th"])]
        if not cells:
            continue
        row = dict(zip(header_cells, cells))
        rows.append(row)
    return rows


def parse_domain_page(html, domain_code, catalog, url):
    """Parse a single control-catalog domain page (e.g. /control-catalog/cybersecurity/ga/)
    by flattening its content blocks and locating control-id / section-label
    text landmarks, since the site is a Next.js app with no <h3>-per-control
    heading structure to rely on."""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()

    body = soup.find("body") or soup

    h1 = body.find("h1")
    domain_name = h1.get_text(strip=True) if h1 else domain_code.upper()

    controls = []
    current = None

    # Walk every element in document order; react to h2 (new control),
    # h3 (section label) and content elements (p / ul / ol / table).
    all_elements = body.find_all(["h1", "h2", "h3", "p", "ul", "ol", "table"])
    active_section = None
    for el in all_elements:
        if el.name == "h1":
            # Left the controls area (e.g. footer heading) -- stop attributing.
            active_section = None
            continue

        if el.name == "h2":
            text = el.get_text(strip=True)
            match = CONTROL_ID_RE.match(text)
            if not match:
                active_section = None
                continue
            if current is not None:
                controls.append(current)
            ctrl_id = f"{match.group(1)}-{match.group(2)}"
            current = {
                "id": ctrl_id,
                "title": match.group(3).strip(),
                "domain": domain_code.upper(),
                "catalog": catalog,
                "statement": "",
                "parameters": [],
                "risk": "",
                "rationale": "",
                "levels": [],
                "sourceUrl": url,
                "retrievedAt": RETRIEVED_AT,
                "_recommendations": "",  # kept internally, folded into statement context if needed
            }
            active_section = "group"
            continue

        if current is None:
            continue

        if el.name == "h3":
            label = el.get_text(strip=True).lower()
            active_section = SECTION_LABELS.get(label)
            continue

        if active_section == "group":
            # "Group: <domain name>" paragraph right after the h2 -- informational only.
            active_section = None
            continue

        if active_section == "parameters" and el.name == "table":
            current["parameters"].extend(parse_parameters_table(el))
            continue

        if active_section in ("statement", "recommendations", "risk", "rationale") and el.name in ("p", "ul", "ol"):
            text = el.get_text(strip=True, separator=" ")
            if not text:
                continue
            if active_section == "recommendations":
                current["_recommendations"] = (current["_recommendations"] + " " + text).strip()
            else:
                current[active_section] = (current[active_section] + " " + text).strip()

    if current is not None:
        controls.append(current)

    for c in controls:
        c["recommendations"] = c.pop("_recommendations")

    return domain_name, controls


def discover_catalogs():
    """Discover catalog slugs (e.g. cybersecurity, dss) from the control-catalog
    index page, instead of assuming which catalogs exist."""
    html = fetch_html(CATALOG_BASE, allow_404=False)
    soup = BeautifulSoup(html, "html.parser")
    catalogs = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        m = re.match(r"^/control-catalog/([a-z0-9-]+)/$", href)
        if m:
            catalogs.add(m.group(1))
    return sorted(catalogs)


def discover_domains(catalog):
    """Discover domain codes for a catalog from that catalog's own index page."""
    url = f"{CATALOG_BASE}{catalog}/"
    html = fetch_html(url, allow_404=False)
    soup = BeautifulSoup(html, "html.parser")
    prefix = f"/control-catalog/{catalog}/"
    codes = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        m = re.match(rf"^{re.escape(prefix)}([a-z0-9]+)/$", href)
        if m:
            codes.add(m.group(1))
    return sorted(codes)


def parse_level_definitions():
    """Parse the real Control Levels section from the /ssp/ page (Level 0/1/2).

    The site nests this section's rows inside a table, so a sibling walk from
    the "Control Levels" <h3> misses them; instead we walk the page's leaf
    text blocks in document order (the same block-flattening approach used
    for control pages) and collect text between this heading and the next
    <h3>.
    """
    html = fetch_html(BASE_URL, allow_404=False)
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()

    body = soup.find("body") or soup
    blocks = list(leaf_blocks(body))

    start = None
    for i, (name, el) in enumerate(blocks):
        if name == "h3" and el.get_text(strip=True).lower() == "control levels":
            start = i
            break

    if start is None:
        raise RuntimeError("Could not find the 'Control Levels' section on the /ssp/ page.")

    levels = []
    pending_level = None
    for name, el in blocks[start + 1:]:
        if name == "h3":
            break
        text = el.get_text(strip=True)
        if not text:
            continue
        if re.match(r"^Level\s+\d+$", text):
            pending_level = text
        elif pending_level and text not in ("Control", "Description"):
            levels.append({"level": pending_level, "description": text, "sourceUrl": BASE_URL})
            pending_level = None

    return levels


def parse_system_type_baseline(html, type_id, url):
    """Parse a /ssp/<type>/ baseline page.

    Unlike the control-catalog domain pages (h2 = control id, h3 = section
    label), these pages nest one level deeper: h2 = domain group heading,
    h3 = control id, h4 = section label. Each control heading is followed by
    a "Group: ..." paragraph and a "Profile Level: <n>" paragraph -- the
    latter's value is a bold-label + trailing text node, so it must be read
    with the tag's full get_text(), not the leaf-block scan (which would
    only yield the bold "Profile Level:" label and drop the value).

    Returns (security_sensitivity, [{"controlId": ..., "profileLevel": ...}]).
    """
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()

    body = soup.find("body") or soup
    elements = body.find_all(["h1", "h2", "h3", "p"])

    security_sensitivity = None
    in_system_characteristics = False
    current_id = None
    entries = []

    for el in elements:
        if el.name == "h1":
            in_system_characteristics = False
            current_id = None
            continue

        if el.name == "h2":
            text = el.get_text(strip=True)
            in_system_characteristics = text.lower() == "system characteristics"
            if current_id is not None:
                raise RuntimeError(f"{type_id}: control {current_id} has no Profile Level before the next section.")
            current_id = None
            continue

        if el.name == "h3":
            if current_id is not None:
                raise RuntimeError(f"{type_id}: control {current_id} has no Profile Level before the next control.")
            text = el.get_text(strip=True)
            match = CONTROL_ID_RE.match(text)
            current_id = f"{match.group(1)}-{match.group(2)}" if match else None
            continue

        if el.name != "p":
            continue

        text = el.get_text(strip=True)
        if in_system_characteristics and text.lower().startswith("security sensitivity level:"):
            security_sensitivity = text.split(":", 1)[1].strip()
            continue

        if current_id is not None and text.lower().startswith("profile level:"):
            level_str = text.split(":", 1)[1].strip()
            if not level_str.isdigit():
                raise RuntimeError(f"{type_id}: control {current_id} has a non-numeric Profile Level '{level_str}'.")
            entries.append({"controlId": current_id, "profileLevel": int(level_str)})
            current_id = None

    if current_id is not None:
        raise RuntimeError(f"{type_id}: control {current_id} has no Profile Level before end of page.")

    return security_sensitivity, entries


def discover_system_types():
    """Discover system-type pages linked from the /ssp/ index page, and parse
    each one's control baseline (control id -> profile level) and security
    sensitivity level."""
    html = fetch_html(BASE_URL, allow_404=False)
    soup = BeautifulSoup(html, "html.parser")
    links = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("/ssp/") and len(href) > len("/ssp/") and href.endswith("/"):
            links.add(href)

    system_types = []
    baselines = []
    for link in sorted(links):
        full_url = urljoin(BASE_URL, link)
        type_html = fetch_html(full_url)
        if type_html is None:
            continue
        type_id = link.strip("/").split("/")[-1]

        soup2 = BeautifulSoup(type_html, "html.parser")
        title_tag = soup2.find("h1") or soup2.find("title")
        name = title_tag.get_text(strip=True) if title_tag else type_id.replace("-", " ").title()
        desc_tag = soup2.find("p")
        description = desc_tag.get_text(strip=True) if desc_tag else ""

        security_sensitivity, entries = parse_system_type_baseline(type_html, type_id, full_url)
        if not entries:
            raise RuntimeError(f"System type {type_id} yielded 0 baseline controls.")

        system_types.append({
            "id": type_id,
            "name": name,
            "description": description,
            "securitySensitivity": security_sensitivity,
            "sourceUrl": full_url,
            "retrievedAt": RETRIEVED_AT,
        })
        for entry in entries:
            baselines.append({
                "systemType": type_id,
                "controlId": entry["controlId"],
                "profileLevel": entry["profileLevel"],
                "sourceUrl": full_url,
                "retrievedAt": RETRIEVED_AT,
            })

    return system_types, baselines


def scrape_ssp_comprehensive():
    print("Starting comprehensive SSP data acquisition...")

    print("\n--- Discovering system types (/ssp/) ---")
    system_types, baselines = discover_system_types()
    print(f"  Found {len(system_types)} system type pages.")

    print("\n--- Discovering catalogs (/control-catalog/) ---")
    catalogs = discover_catalogs()
    if not catalogs:
        raise RuntimeError("No catalogs discovered from the control-catalog index page.")
    print(f"  Discovered catalogs: {catalogs}")

    domains = []
    all_controls = []
    domains_by_catalog = {}

    for catalog in catalogs:
        codes = discover_domains(catalog)
        domains_by_catalog[catalog] = codes
        print(f"\n--- Catalog: {catalog} ({len(codes)} domains) ---")
        for code in codes:
            url = f"{CATALOG_BASE}{catalog}/{code}/"
            print(f"  -> Scraping domain: {catalog}/{code}")
            domain_html = fetch_html(url)
            if domain_html is None:
                print("     [!] Skipped (404)")
                continue

            domain_name, controls = parse_domain_page(domain_html, code, catalog, url)
            domains.append({
                "id": code.upper(),
                "catalog": catalog,
                "name": domain_name,
                "sourceUrl": url,
                "retrievedAt": RETRIEVED_AT,
            })
            print(f"     {len(controls)} controls extracted")
            all_controls.extend(controls)

    print("\n--- Parsing Control Levels (/ssp/) ---")
    level_definitions = parse_level_definitions()
    print(f"  Found {len(level_definitions)} level definitions: {[l['level'] for l in level_definitions]}")

    print("\n--- Validating baseline membership against the catalog ---")
    known_control_ids = {c["id"] for c in all_controls}
    for entry in baselines:
        if entry["controlId"] not in known_control_ids:
            raise RuntimeError(
                f"Baseline for {entry['systemType']} lists control {entry['controlId']}, "
                "which is not in any discovered control-catalog domain."
            )
    print(f"  {len(baselines)} baseline memberships validated against {len(known_control_ids)} catalog controls.")

    return {
        "controls": all_controls,
        "domains": domains,
        "system_types": system_types,
        "level_definitions": level_definitions,
        "baselines": baselines,
        "catalogs": catalogs,
        "domains_by_catalog": domains_by_catalog,
    }


import datetime as _dt
RETRIEVED_AT = _dt.datetime.now(_dt.timezone.utc).isoformat()

ssp_data = scrape_ssp_comprehensive()

print("\nScraping complete.")
print(f"  - Controls: {len(ssp_data['controls'])}")
print(f"  - Domains: {len(ssp_data['domains'])}")
print(f"  - System Types: {len(ssp_data['system_types'])}")
print(f"  - Level Definitions: {len(ssp_data['level_definitions'])}")
print(f"  - Baseline memberships: {len(ssp_data['baselines'])}")
print(f"  - HTTP requests made: {_stats['requests']}, 404s encountered: {_stats['not_found']}")


In [ ]:
import hashlib
import datetime as _dt


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def write_dataset(new_data, stem, snapshot_dir):
    """Write a dataset both to the dated snapshot folder and to the flat
    '-latest' file, returning the flat file's path for the manifest."""
    snap_path = snapshot_dir / f"{stem}.json"
    latest_path = OUTPUT_DIR / f"SSP-{stem}-latest.json"

    with open(snap_path, "w", encoding="utf-8") as f:
        json.dump(new_data, f, indent=2, ensure_ascii=False)

    with open(latest_path, "w", encoding="utf-8") as f:
        json.dump(new_data, f, indent=2, ensure_ascii=False)

    print(f"  Wrote {stem}: {len(new_data)} records -> {snap_path.name} and {latest_path.name}")
    return latest_path


# ------------------------------------------------------------
# Integrity checks before writing anything out
# ------------------------------------------------------------
if not ssp_data["controls"]:
    raise RuntimeError("Zero controls scraped in total -- refusing to write output.")

counts_per_domain = {}
for c in ssp_data["controls"]:
    counts_per_domain[c["domain"]] = counts_per_domain.get(c["domain"], 0) + 1

for domain in ssp_data["domains"]:
    code = domain["id"]
    if counts_per_domain.get(code, 0) == 0:
        raise RuntimeError(f"Domain {domain['catalog']}/{code} yielded 0 controls -- parser likely broken for this page.")

counts_per_catalog = {}
for c in ssp_data["controls"]:
    counts_per_catalog[c["catalog"]] = counts_per_catalog.get(c["catalog"], 0) + 1

# ------------------------------------------------------------
# Fill each control's "levels" from the baselines: which system types
# include it, and at what profile level.
# ------------------------------------------------------------
levels_by_control = {}
for entry in ssp_data["baselines"]:
    levels_by_control.setdefault(entry["controlId"], []).append(
        {"systemType": entry["systemType"], "profileLevel": entry["profileLevel"]}
    )

for c in ssp_data["controls"]:
    c["levels"] = levels_by_control.get(c["id"], [])

controls_in_no_baseline = [c["id"] for c in ssp_data["controls"] if not c["levels"]]
print(f"Controls in no baseline: {len(controls_in_no_baseline)}")

# Per system-type counts: total controls, and counts per profile level.
counts_per_system_type = {}
for entry in ssp_data["baselines"]:
    st = counts_per_system_type.setdefault(entry["systemType"], {"total": 0, "byLevel": {}})
    st["total"] += 1
    st["byLevel"][str(entry["profileLevel"])] = st["byLevel"].get(str(entry["profileLevel"]), 0) + 1

print("\nBaseline membership per system type:")
for type_id in sorted(counts_per_system_type):
    st = counts_per_system_type[type_id]
    by_level = ", ".join(f"L{lvl}={cnt}" for lvl, cnt in sorted(st["byLevel"].items()))
    print(f"  {type_id}: {st['total']} controls ({by_level})")

# ------------------------------------------------------------
# Snapshot folder: dated (site exposes no separate version number on
# these pages, so the retrieval date is the version key).
# ------------------------------------------------------------
snapshot_name = _dt.datetime.now(_dt.timezone.utc).strftime("%Y-%m-%d")
snapshot_dir = OUTPUT_DIR / snapshot_name
snapshot_dir.mkdir(parents=True, exist_ok=True)
print(f"Snapshot folder: {snapshot_dir}")

output_files = {}
output_files["controls"] = write_dataset(ssp_data["controls"], "controls", snapshot_dir)
output_files["domains"] = write_dataset(ssp_data["domains"], "domains", snapshot_dir)
output_files["system_types"] = write_dataset(ssp_data["system_types"], "system-types", snapshot_dir)
output_files["level_definitions"] = write_dataset(ssp_data["level_definitions"], "level-definitions", snapshot_dir)
output_files["baselines"] = write_dataset(ssp_data["baselines"], "baselines", snapshot_dir)

# ------------------------------------------------------------
# Manifest
# ------------------------------------------------------------
manifest_files = {}
for stem, path in output_files.items():
    manifest_files[str(path.relative_to(OUTPUT_DIR)).replace("\\", "/")] = {
        "sha256": sha256_of(path),
        "records": len(ssp_data[stem]),
    }
    snap_path = snapshot_dir / path.name.replace("-latest", "")
    if snap_path.exists():
        manifest_files[str(snap_path.relative_to(OUTPUT_DIR)).replace("\\", "/")] = {
            "sha256": sha256_of(snap_path),
            "records": len(ssp_data[stem]),
        }

manifest = {
    "sourceBaseUrl": BASE_URL,
    "controlCatalogBaseUrl": CATALOG_BASE,
    "retrievedAt": RETRIEVED_AT,
    "snapshotFolder": snapshot_name,
    "discoveredDomains": [d["id"] for d in ssp_data["domains"]],
    "discoveredCatalogs": ssp_data["catalogs"],
    "domainsByCatalog": ssp_data["domains_by_catalog"],
    "countsPerDomain": counts_per_domain,
    "countsPerCatalog": counts_per_catalog,
    "countsPerSystemType": counts_per_system_type,
    "controlsInNoBaselineCount": len(controls_in_no_baseline),
    "totalControls": len(ssp_data["controls"]),
    "notFoundCount": _stats["not_found"],
    "httpRequests": _stats["requests"],
    "files": manifest_files,
}

if manifest["totalControls"] == 0:
    raise RuntimeError("Manifest total control count is 0 -- refusing to finalize.")

manifest_path = OUTPUT_DIR / "manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f"\nManifest written to {manifest_path}")
print(f"Snapshot folder : {manifest['snapshotFolder']}")
print(f"Retrieved at    : {manifest['retrievedAt']}")
print(f"Total controls  : {manifest['totalControls']}")
print(f"404s            : {manifest['notFoundCount']}")
print(f"HTTP requests   : {manifest['httpRequests']}")
print("Counts per catalog:", json.dumps(manifest["countsPerCatalog"], indent=2))
print("\nFiles (sha256):")
for fname, info in manifest["files"].items():
    print(f"  {fname}: {info['records']} records, sha256={info['sha256']}")


In [ ]:
from collections import OrderedDict
import json

def infer_structure(obj):
    """
    Extract the recursive structure of a JSON document.
    """
    if isinstance(obj, dict):
        return {key: infer_structure(value) for key, value in obj.items()}

    if isinstance(obj, list):
        if not obj:
            return ["unknown"]

        if all(isinstance(item, dict) for item in obj):
            merged = {}
            for item in obj:
                item_structure = infer_structure(item)
                for key, structure in item_structure.items():
                    if key not in merged:
                        merged[key] = structure
            return [merged]

        types = sorted({type(item).__name__ for item in obj})
        return [types[0] if len(types) == 1 else types]

    if obj is None:
        return "null"
    if isinstance(obj, bool):
        return "boolean"
    if isinstance(obj, int):
        return "integer"
    if isinstance(obj, float):
        return "number"
    if isinstance(obj, str):
        return "string"

    return type(obj).__name__

files_to_analyze = {
    "Controls": OUTPUT_DIR / "SSP-controls-latest.json",
    "Domains": OUTPUT_DIR / "SSP-domains-latest.json",
    "System Types": OUTPUT_DIR / "SSP-system-types-latest.json",
    "Level Definitions": OUTPUT_DIR / "SSP-level-definitions-latest.json"
}

structures = {}

for name, filepath in files_to_analyze.items():
    print(f"Analyzing {name} structure from {filepath.name}...")
    if filepath.exists() and filepath.stat().st_size > 2: # Check if not empty []
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        structures[name] = infer_structure(data)
    else:
        structures[name] = "Empty or missing"

print("\n" + "="*70)
print("INFERRED JSON STRUCTURES")
print("="*70)
print(json.dumps(structures, indent=2, ensure_ascii=False))

In [ ]:
print("Counts per domain:")
for domain_id in sorted(counts_per_domain):
    print(f"  {domain_id}: {counts_per_domain[domain_id]}")

ga_controls = [c for c in ssp_data["controls"] if c["domain"] == "GA"]
if ga_controls:
    print(f"\nGenerative AI (GA) domain present: {len(ga_controls)} controls")
    ga_ids = sorted((c["id"] for c in ga_controls), key=lambda x: int(x.split("-")[1]))
    print("  IDs:", ", ".join(ga_ids))
else:
    print("\nGenerative AI (GA) domain NOT found in this run's discovered domains.")

print("\nSample controls (up to 3):")
for c in ssp_data["controls"][:3]:
    print(json.dumps(c, indent=2, ensure_ascii=False))
    print("-" * 70)


# Field coverage: how many controls carry a non-empty value per field,
# so an empty field can be told apart from a parser gap.
_ctrl = json.loads((OUTPUT_DIR / 'SSP-controls-latest.json').read_text(encoding='utf-8'))
print('\nField coverage (non-empty / total):')
for _k in sorted({k for c in _ctrl for k in c}):
    print(f'  {_k}: {sum(1 for c in _ctrl if c.get(_k))} / {len(_ctrl)}')
